In [153]:
import pandas as pd

## Data Load

In [154]:
df = pd.read_csv("gk_qna_dataset.csv")
df

,question,answer
0,"What is the capital of ""France""?",Paris
1,"What is the capital of ""Germany""?",Berlin
2,"What is the capital of ""Italy""?",Rome
3,"What is the capital of ""Spain""?",Madrid
4,"What is the capital of ""Japan""?",Tokyo
...,...,...
166,"Can you tell me, ""Which bird cannot fly""?",Ostrich
167,"I was wondering, ""What is 'H2O' commonly called""?",Water
168,"I was wondering, ""Which planet is known as the...",Mars
169,"Can you tell me, ""What is the main religion in...",Islam


In [155]:
df['question'][0]

'What is the capital of "France"?'

## Data Preprocessing

In [156]:
import re
def tokenization(text):
  text = text.lower()
  #remove quotation
  text = re.sub( r"[\"']","",text)
  #remove space and punctutaion
  text = re.sub(r"[^a-z0-9\s]"," ",text)
  #remove extra spaces
  text = re.sub(r"\s+"," ",text).strip()
  tokens = text.split()
  return tokens

In [157]:
print(tokenization("Don't stop, it's \"great\" "))

['dont', 'stop', 'its', 'great']


In [158]:
#building vocabulary
vocab = {'<UNK>':0}
def build_vocab(row):
  tokenized_question = tokenization(row['question'])
  tokenized_answer = tokenization(row['answer'])
  tokenized_merged = tokenized_question + tokenized_answer

  for token in tokenized_merged:
    if token not in vocab:
      vocab[token]=len(vocab)


In [159]:
df.apply(build_vocab,axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
166,None
167,None
168,None
169,None


In [160]:
vocab

{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'italy': 10,
 'rome': 11,
 'spain': 12,
 'madrid': 13,
 'japan': 14,
 'tokyo': 15,
 'canada': 16,
 'ottawa': 17,
 'brazil': 18,
 'brasilia': 19,
 'australia': 20,
 'canberra': 21,
 'india': 22,
 'new': 23,
 'delhi': 24,
 'china': 25,
 'beijing': 26,
 'russia': 27,
 'moscow': 28,
 'united': 29,
 'states': 30,
 'washington': 31,
 'dc': 32,
 'mexico': 33,
 'city': 34,
 'egypt': 35,
 'cairo': 36,
 'turkey': 37,
 'ankara': 38,
 'argentina': 39,
 'buenos': 40,
 'aires': 41,
 'south': 42,
 'korea': 43,
 'seoul': 44,
 'indonesia': 45,
 'jakarta': 46,
 'pakistan': 47,
 'islamabad': 48,
 'bangladesh': 49,
 'dhaka': 50,
 'nepal': 51,
 'kathmandu': 52,
 'sri': 53,
 'lanka': 54,
 'colombo': 55,
 'thailand': 56,
 'bangkok': 57,
 'malaysia': 58,
 'kuala': 59,
 'lumpur': 60,
 'vietnam': 61,
 'hanoi': 62,
 'uae': 63,
 'abu': 64,
 'dhabi': 65,
 'iran': 66,
 'tehran': 67,
 'iraq

In [161]:
#convert text to index
def text_to_indices(text,vocab):
  indexed_text = []

  for token in tokenization(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])

  return indexed_text

In [162]:
print(text_to_indices("What is the capital of dhaka",vocab))

[1, 2, 3, 4, 5, 50]


## Dataset and DataLoader

In [163]:
import torch
from torch.utils.data import Dataset,DataLoader

In [164]:
class gkDataset(Dataset):
  def __init__(self,df,vocab):
    super().__init__()
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self,index):
    numerical_question = text_to_indices(self.df.iloc[index]['question'],self.vocab)
    numerical_answer = text_to_indices(self.df.iloc[index]['answer'],self.vocab)
    return torch.tensor(numerical_question),torch.tensor(numerical_answer)


In [165]:
dataset = gkDataset(df,vocab)

In [166]:
dataloader = DataLoader(dataset,batch_size=1,shuffle=True)#batch size 1 don't need padding

In [167]:
for question,answer in dataloader:
  print(question,answer[0])

tensor([[237,   1, 238, 239,  93, 215, 216, 213]]) tensor([217])
tensor([[240, 232, 241, 242,  93, 197,   2, 200]]) tensor([201])
tensor([[240, 232, 241, 242,  93, 100,   2, 101, 102,   3, 103, 100]]) tensor([104])
tensor([[237,   1, 238, 239,   1, 131, 145, 133, 121]]) tensor([146, 147, 148])
tensor([[240, 232, 241, 242,   1, 131, 132, 133, 121]]) tensor([134, 130, 135])
tensor([[ 1,  2,  3,  4,  5, 70, 71]]) tensor([72])
tensor([[1, 2, 3, 4, 5, 6]]) tensor([7])
tensor([[230, 231,  93, 179,   2, 101, 102,   3, 180,   5, 181, 182]]) tensor([14])
tensor([[234, 235, 236,   1,   2, 124, 120, 121]]) tensor([125, 126])
tensor([[ 93, 209,   2, 210, 211, 212, 213]]) tensor([214])
tensor([[230, 231,  93, 202, 203, 204]]) tensor([205])
tensor([[ 79, 114,   3, 115,   5, 116]]) tensor([117, 118])
tensor([[230, 231,  79,  84,   3,  85]]) tensor([86, 87, 88])
tensor([[ 93, 215, 216, 218,  95, 232, 233]]) tensor([219])
tensor([[240, 232, 241, 242,   1,   2, 141, 142]]) tensor([143, 144])
tensor([[ 9

## RNN Architechture Implementation

In [168]:
import torch.nn as nn

class simpleRNN(nn.Module):
  def __init__(self,vocab_size,embedding_dim=50,hidden_size=64):
    super().__init__()

    self.embedding = nn.Embedding(vocab_size,embedding_dim)
    self.rnn = nn.RNN(input_size= embedding_dim,hidden_size= hidden_size,batch_first=True)
    self.fc = nn.Linear(hidden_size,vocab_size)

  def forward(self,question):
    embedded = self.embedding(question)
    _,final = self.rnn(embedded)

    return self.fc(final.squeeze(0))

In [169]:
model = simpleRNN(vocab_size=len(vocab))

In [170]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)
epochs = 20

In [171]:
for epoch in range(epochs):
  total_loss = 0
  for question,answer in dataloader:
    optimizer.zero_grad()
    output = model(question)
    target = answer[0][0].unsqueeze(0)
    loss = criterion(output,target)
    loss.backward()
    optimizer.step()
    total_loss += loss.item()

  print(f"Epoch: {epoch+1}, Loss: {total_loss:4f}")

Epoch: 1, Loss: 855.283441
Epoch: 2, Loss: 541.002852
Epoch: 3, Loss: 358.212555
Epoch: 4, Loss: 259.063155
Epoch: 5, Loss: 198.025172
Epoch: 6, Loss: 153.355504
Epoch: 7, Loss: 116.314740
Epoch: 8, Loss: 87.447164
Epoch: 9, Loss: 63.325664
Epoch: 10, Loss: 46.980635
Epoch: 11, Loss: 33.942947
Epoch: 12, Loss: 25.125175
Epoch: 13, Loss: 19.228305
Epoch: 14, Loss: 14.981111
Epoch: 15, Loss: 12.048647
Epoch: 16, Loss: 9.792892
Epoch: 17, Loss: 8.148531
Epoch: 18, Loss: 6.863103
Epoch: 19, Loss: 5.865191
Epoch: 20, Loss: 5.038886


In [172]:
def predict(question_text,model,vocab,threshold = 0.5):
  model.eval()
  with torch.no_grad():
    indices = text_to_indices(question_text,vocab)
    x = torch.tensor(indices).unsqueeze(0)

    output = model(x)

    probs = torch.nn.functional.softmax(output,dim=1)

    value,idx = torch.max(probs,dim=1)
    if value.item() <threshold:
      print("I don't know")
      return None

    print(list(vocab.keys())[idx])

In [173]:
predict("What is the capital of Japan ?", model, vocab)

tokyo


In [174]:
# Test
tests = [
    "What is the capital of France?",
    "What is H2O commonly called?",
    "What is red planet?",
    "Which bird cannot fly?",
    "What is the capital of Japan ?",

]
for q in tests:
    print(f"{q:45} → ",end="")
    predict(q, model, vocab)

What is the capital of France?                → paris
What is H2O commonly called?                  → water
What is red planet?                           → mars
Which bird cannot fly?                        → ostrich
What is the capital of Japan ?                → tokyo
